# SDM invullen

### Imports

In [ ]:
import sqlite3
import pandas as pd
import os
from datetime import datetime

### Verbinding maken met SDM

In [ ]:
SDM = sqlite3.connect("../../Database/SDM2.db")
cursor = SDM.cursor()

#alle databases verbinding 
dbs = {
    "Accessoireverkoop": sqlite3.connect("../../Database/BikeToDrive_1_Accessoireverkoop.db"),
    "Fietsverkoop": sqlite3.connect("../../Database/BikeToDrive_2_Fietsverkoop.db"),
    "Onderhoud": sqlite3.connect("../../Database/BikeToDrive_3_Onderhoud.db"),
    "Accessoire_Inkoop": sqlite3.connect("../../Database/BikeToDrive_4_Accessoire_Inkoop.db"),
    "Fiets_Inkoop": sqlite3.connect("../../Database/BikeToDrive_5_Fiets_Inkoop.db"),
}

Logging

In [ ]:
log_list = []

def log(source, table, action, key, expected, actual, status, note):
    log_list.append({
        "source": source,
        "table": table,
        "action": action,
        "key": key,
        "expected": expected,
        "actual": actual,
        "status": status,
        "note": note
    })

### Load Data


In [ ]:

#Alle tabellen selecteren 
def get_tables(conn):
    #read
    return pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
        #lijst met alle tabel namen 
    )['name'].tolist()


#laad tabel 
def load_db(conn, name):
    data = {}

    # For alle tabel in dit db 
    for t in get_tables(conn):

        #read => into dataframe
        df = pd.read_sql_query(f"SELECT * FROM {t}", conn)

        #unique key 
        key = f"{name}_{t}"

        # Store DataFrame in dictionary
        data[key] = df

        #controleren of het ingeladen is
        print(f"Loaded: {key}")
    return data

#combinneer alle data => 1 dictionary
all_data = {}

for name, conn in dbs.items():
    all_data.update(load_db(conn, name))


Dit zorgt er voor dat we later makelijker dingen kunnen doen zoals:
1. automatisch tabellen maken
2. automatisch tabellen leegmaken
3. automatisch data laden

### Reset_knop
a.	Alle tabellen uit het SDM leegmaken. Dit is dus een soort “reset-knop” voor de invulling van je SDM-tabellen.

In [ ]:
#reset functie 
def reset_sdm(conn):
    #SDM => alle tabel name
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )['name'].tolist()

    #reverse => foreign key issue oplosen 
    for t in reversed(tables):
        try:
            conn.execute(f"DELETE FROM {t}")
            print(f"Cleared: {t}")
        except:
            pass

    conn.commit()

#run
reset_sdm(SDM)


### Mapping 

In [ ]:
#Mapping => tussen SDM tables en source entities 
SDM_MAPPING = {
    #entity
    "Leverancier": [
        "Accessoireverkoop_Leverancier", #tabel
        "Accessoire_Inkoop_Leverancier"
    ],

    "Filiaal": [
        "Accessoireverkoop_Filiaal",
        "Fietsverkoop_Filiaal",
        "Onderhoud_Filiaal"
    ],

    "Fabrikant": [
        "Fietsverkoop_Fabrikant",
        "Onderhoud_Fabrikant",
        "Fiets_Inkoop_Fabrikant"
    ],

    "Klant": [
        "Accessoireverkoop_Klant",
        "Fietsverkoop_Klant"
    ],

    "Monteur": [
        "Accessoireverkoop_Monteur",
        "Fietsverkoop_Monteur",
        "Onderhoud_Monteur"
    ],

    "Accessoire": [
        "Accessoireverkoop_Accessoire",
        "Accessoire_Inkoop_Accessoire"
    ],

    "Fiets": [
        "Fietsverkoop_Fiets",
        "Onderhoud_Fiets",
        "Fiets_Inkoop_Fiets"
    ]
}


insert to sdm

In [ ]:
def insert(df, table, conn):
    try:
        df.to_sql(table, conn, if_exists="append", index=False)

        log(
            "Source → SDM",
            table,
            "INSERT",
            "-",
            f"{len(df)} rows expected",
            f"{len(df)} rows inserted",
            "SUCCESS",
            "Bulk load into SDM"
        )

        conn.commit()

    except Exception as e:
        log(
            "Source → SDM",
            table,
            "INSERT FAILED",
            "-",
            "expected insert",
            str(e),
            "FAILED",
            "Database write error"
        )
        print(f"ERROR inserting {table}: {e}")



Load dimensions


In [ ]:
def load_dimensions(mapping, data, conn):

    for entity, tables in mapping.items():

        for table in tables:

            source_key = None

            # find matching source table
            for k in data.keys():
                if k.endswith(f"_{entity}"):
                    source_key = k

            # If found → insert into SDM
            if source_key:

                df = data[source_key]   # ✅ FIX: define df here

                insert(df, table, conn)

                log(
                    "SDM → SDM",
                    table,
                    "DIM LOAD",
                    "-",
                    f"{len(df)} rows expected",
                    f"{len(df)} rows loaded",
                    "SUCCESS",
                    f"{entity} loaded into SDM"
                )

            else:

                log(
                    "SDM → SDM",
                    table,
                    "DIM LOAD",
                    "-",
                    "expected data",
                    "missing source",
                    "FAILED",
                    f"Missing entity {entity}"
                )

                print(f"Missing: {entity}")


Load facts

In [ ]:
# Mapping => feiten
fact_tables = {
    "Onderhoud": "Onderhoud_Onderhoud",
    "Accessoire_Inkoop": "Accessoire_Inkoop_Accessoire_Inkoop",
    "Fiets_Inkoop": "Fiets_Inkoop_Fiets_Inkoop",
    "Accessoireverkoop_Accessoire_Verkoop": "Accessoireverkoop_Accessoire_Verkoop",
    "Fietsverkoop_Fiets_Verkoop": "Fietsverkoop_Fiets_Verkoop"
}


# for target, source in fact_tables.items():
#      # controleren of tabel bestaat
#     if source in all_data:
#         # Insert into SDM
#         all_data[source].to_sql(target, SDM, if_exists="append", index=False)
#         print(f"FACT LOADED → {target}")

for target, source in fact_tables.items():

    if source in all_data:

        df = all_data[source]

        df.to_sql(target, SDM, if_exists="append", index=False)

        log(
            "Source → SDM",
            target,
            "FACT LOAD",
            "-",
            f"{len(df)} rows expected",
            f"{len(df)} rows loaded",
            "SUCCESS",
            "Fact table loaded"
        )

        print(f"FACT LOADED → {target}")

    else:

        log(
            "Source → SDM",
            target,
            "FACT LOAD",
            "-",
            "expected data",
            "missing source",
            "FAILED",
            "Fact source missing"
        )



In [ ]:
SDM.commit()


controleren of er daadwerkelijk rijen bestaan => er is een sdm

In [ ]:
#row count per tabel controleren
def check_counts(conn):
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )['name'].tolist()

    for t in tables:
        count = pd.read_sql_query(f"SELECT COUNT(*) as c FROM {t}", conn)['c'][0]
        print(f"{t}: {count}")


check_counts(SDM)


In [ ]:
#SDM tabellen namen zien

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    SDM
)

print(tables)


Logbook bijhouden

In [ ]:
# log_df = pd.DataFrame(log_list)
# log_df.to_excel("sdm_logbook.xlsx", index=False)
# print("SDM logbook saved")

file = "dwh_logbook.xlsx"

log_df = pd.DataFrame(log_list)

if os.path.exists(file):
    old = pd.read_excel(file)
    log_df = pd.concat([old, log_df], ignore_index=True)

log_df.to_excel(file, index=False)